In [43]:
from dotenv import dotenv_values
import requests
import time
import re
import pandas as pd
from io import StringIO
import sys
from ipywidgets import interactive, fixed, interact_manual,Accordion
import ipywidgets as widgets

# import custom modules
sys.path.append('../../utils/')
import data_paths

In [44]:
# Available road gradients (idgrad):
# 30: 0%
# 32: +/-2%
# 34: +/-4%
# 36: +/-6%
# 54: -6%
# 56: -4%
# 58: -2%
# 62: +2%
# 64: +4%
# 66: +6%
def select_gradient(grad_0=True, grad_pm2=True, grad_pm4=True, grad_pm6=True,
                    grad_m6=True, grad_m4=True, grad_m2=True,
                    grad_p2=True, grad_p4=True, grad_p6=True):
    gradient_map = {
    "30": grad_0,
    "32": grad_pm2,
    "34": grad_pm4,
    "36": grad_pm6,
    "54": grad_m6,
    "56": grad_m4,
    "58": grad_m2,
    "62": grad_p2,
    "64": grad_p4,
    "66": grad_p6
    }


    return ",".join([grad for grad, enabled in gradient_map.items() if enabled])

# Pollutants: HC, CO, NOx, NO2, CO2(rep), CO2(total), PM10-ex, PN23-ex,
# CH4, NHMC, Pb, SO2, N2O, NH3, Zn-ex, Zn-nx, Cd-ex, Cd-nx, PM10-nx,
# Benzene, Toluene, Xylene, FC, EC, PM2.5-ex, BC-ex, PM2.5-nx, BC-nx,
# CO2e, WE-pos, HCHO, CH3CHO, HNCO, HNO2, PM10-nx-tyre, PM10-nx-brake,
# PM10-nx-road, PM10-nx-resusp, PM2.5-nx-tyre, PM2.5-nx-brake,
# PM2.5-nx-road, PM2.5-nx-resusp, PN23-nx-brake, PN23-nx-road,
# PN23-nx-resusp, PN23-nx
def select_pollutants(hc=True, co=True, nox=True, no2=True, co2_rep=True, co2_total=True,
                      pm10_ex=True, pn23_ex=True, ch4=True, nmhc=True, pb=True,
                      so2=True, n2o=True, nh3=True, zn_ex=True, zn_nx=True,
                      cd_ex=True, cd_nx=True, pm10_nx=True, benzene=True,
                      toluene=True, xylene=True, fc=True, ec=True,
                      pm25_ex=True, bc_ex=True, pm25_nx=True, bc_nx=True,
                      co2e=True, we_pos=True, hcho=True, ch3cho=True,
                      hnco=False, hno2=False,
                      pm10_nx_tyre=False, pm10_nx_brake=False,
                      pm10_nx_road=False, pm10_nx_resusp=False,
                      pm25_nx_tyre=False, pm25_nx_brake=False,
                      pm25_nx_road=False, pm25_nx_resusp=False,
                      pn23_nx_brake=False, pn23_nx_road=False,
                      pn23_nx_resusp=False, pn23_nx=False):

    pollutant_map = {
        "HC":               hc,
        "CO":               co,
        "NOx":              nox,
        "NO2":              no2,
        "CO2(rep)":         co2_rep,
        "CO2(total)":       co2_total,
        "PM10-ex":          pm10_ex,
        "PN23-ex":          pn23_ex,
        "CH4":              ch4,
        "NMHC":             nmhc,
        "Pb":               pb,
        "SO2":              so2,
        "N2O":              n2o,
        "NH3":              nh3,
        "Zn-ex":            zn_ex,
        "Zn-nx":            zn_nx,
        "Cd-ex":            cd_ex,
        "Cd-nx":            cd_nx,
        "PM10-nx":          pm10_nx,
        "Benzene":          benzene,
        "Toluene":          toluene,
        "Xylene":           xylene,
        "FC":               fc,
        "EC":               ec,
        "PM2.5-ex":         pm25_ex,
        "BC-ex":            bc_ex,
        "PM2.5-nx":         pm25_nx,
        "BC-nx":            bc_nx,
        "CO2e":             co2e,
        "WE-pos":           we_pos,
        "HCHO":             hcho,
        "CH3CHO":           ch3cho,
        "HNCO":             hnco,
        "HNO2":             hno2,
        "PM10-nx-tyre":     pm10_nx_tyre,
        "PM10-nx-brake":    pm10_nx_brake,
        "PM10-nx-road":     pm10_nx_road,
        "PM10-nx-resusp":   pm10_nx_resusp,
        "PM2.5-nx-tyre":    pm25_nx_tyre,
        "PM2.5-nx-brake":   pm25_nx_brake,
        "PM2.5-nx-road":    pm25_nx_road,
        "PM2.5-nx-resusp":  pm25_nx_resusp,
        "PN23-nx-brake":    pn23_nx_brake,
        "PN23-nx-road":     pn23_nx_road,
        "PN23-nx-resusp":   pn23_nx_resusp,
        "PN23-nx":          pn23_nx,
    }
    
    return ",".join([name for name, enabled in pollutant_map.items() if enabled])
#1= PC, 2=LCV, 14=HGV, 6=Coach, 7=Bus, 9=Motorcycle
def select_vehicles(pc = True, lcv = True, hgv = True, coach = True, bus = False, motorcycle = True):
    vehicle_map = {
        "1": pc,
        "2": lcv,
        "14": hgv,
        "6": coach,
        "7": bus,
        "9": motorcycle
    }
    return ",".join([name for name, enabled in vehicle_map.items() if enabled])

    # Aggregated traffic situation pattern: idtsgrad
    # For Germany:
    # --- Current (UBA) ---
    #   521: D Ø-MW UBA 2024        (Motorway only)
    #   522: D Ø-Rural UBA 2024     (Rural only)
    #   523: D Ø-Urban UBA 2024     (Urban only)
    #   524: D Ø UBA 2024           (All road categories)
    #
    #   421: D Ø-MW UBA 2023
    #   422: D Ø-Rural UBA 2023
    #   423: D Ø-Urban UBA 2023
    #   424: D Ø UBA 2023
    #
    #   332: D Ø UBA 2022
    #   333: D Ø-MW UBA 2022
    #   334: D Ø-Rural UBA 2022
    #   335: D Ø-Urban UBA 2022
    #
    #   221: D Ø-MW UBA 2021
    #   222: D Ø-Rural UBA 2021
    #   223: D Ø-Urban UBA 2021
    #   224: D Ø UBA 2021
    #   324: D Ø UBA 2021 detailed
    #   328: D Ø-MW UBA 2021 korr
    #   329: D Ø-Rural UBA 2021 korr
    #   330: D Ø-Urban UBA 2021 korr
    #   331: D Ø UBA 2021 korr
    #
    # --- Outdated (IFEU, Nov 2009) ---
    #   121: Germany Motorway
    #   122: Germany Rural
    #   123: Germany Urban
    #   124: Germany all Road Categories
def select_tsgrad(
    d_mw_2024         = True,
    d_rural_2024      = True,
    d_urban_2024      = True,
    d_all_2024        = True,

    d_mw_2023         = False,
    d_rural_2023      = False,
    d_urban_2023      = False,
    d_all_2023        = False,

    d_all_2022        = False,
    d_mw_2022         = False,
    d_rural_2022      = False,
    d_urban_2022      = False,

    d_mw_2021         = False,
    d_rural_2021      = False,
    d_urban_2021      = False,
    d_all_2021        = False,
    d_all_2021_det    = False,
    d_mw_2021_korr    = False,
    d_rural_2021_korr = False,
    d_urban_2021_korr = False,
    d_all_2021_korr   = False,

    de_mw_ifeu        = False,
    de_rural_ifeu     = False,
    de_urban_ifeu     = False,
    de_all_ifeu       = False,
):
    tsgrad_map = {
        "521": d_mw_2024,
        "522": d_rural_2024,
        "523": d_urban_2024,
        "524": d_all_2024,

        "421": d_mw_2023,
        "422": d_rural_2023,
        "423": d_urban_2023,
        "424": d_all_2023,

        "332": d_all_2022,
        "333": d_mw_2022,
        "334": d_rural_2022,
        "335": d_urban_2022,

        "221": d_mw_2021,
        "222": d_rural_2021,
        "223": d_urban_2021,
        "224": d_all_2021,
        "324": d_all_2021_det,
        "328": d_mw_2021_korr,
        "329": d_rural_2021_korr,
        "330": d_urban_2021_korr,
        "331": d_all_2021_korr,

        "121": de_mw_ifeu,
        "122": de_rural_ifeu,
        "123": de_urban_ifeu,
        "124": de_all_ifeu,
    }

    return ",".join([idtsgrad for idtsgrad, enabled in tsgrad_map.items() if enabled])
#   1: Rural
#   2: Urban
def select_indv_scen_area(rural = True, urban = True):
    scen_area_map = {
        "1": rural,
        "2": urban
    }

    return ",".join([name for name, enabled in scen_area_map.items() if enabled])

#   10: Motorway-Nat.
#   11: Motorway-City
#   12: Semi-Motorway
#   20: Primary-nat. non-motorway
#   21: Primary-city non-motorway
#   30: Distributor/Secondary
#   31: Distributor/Secondary (sinuous)
#   40: Local/Collector
#   41: Local/Collector (sinuous)
#   50: Access-residential
#
def select_indv_scen_roadtype(
    motorway_nat        = True,    # 10
    motorway_city       = False,   # 11
    semi_motorway       = False,   # 12
    primary_nat         = True,    # 20
    primary_city        = True,    # 21
    distributor         = True,    # 30
    distributor_sin     = False,   # 31
    local               = True,    # 40
    local_sin           = False,   # 41
    access_residential  = True,    # 50
):
    roadtype_map = {
        "10": motorway_nat,
        "11": motorway_city,
        "12": semi_motorway,
        "20": primary_nat,
        "21": primary_city,
        "30": distributor,
        "31": distributor_sin,
        "40": local,
        "41": local_sin,
        "50": access_residential,
    }

    return ",".join([code for code, enabled in roadtype_map.items() if enabled])

#TODO         # idspeedlimit (speed limit in km/h):
        #   03: 30,  04: 40,  05: 50,  06: 60,  07: 70,  08: 80
        #   09: 90, 10: 100, 11: 110, 12: 120, 13: 130, 14: >130
        #
        # idlos (level of service):
        #   1: Freeflow
        #   2: Heavy
        #   3: Saturated
        #   4: Stop+go
        #   5: Stop+go_II
        #
        # Example: 110081 -> area=Rural, Motorway-Nat., speedlimit=80, LOS=Freeflow
        #"idts": "110081,110082",
# Available ambient condition patterns (idpatternambientcond):
        # --- Simple patterns (averaged dimensions) ---
# ID  : label                  : description
#   1 : ØGermany               : Ø trip lengths, Ø parking times
#  11 : Ø/spring               : Ø trip lengths, Ø parking times
#  12 : Ø/summer               : Ø trip lengths, Ø parking times
#  13 : Ø/autumn               : Ø trip lengths, Ø parking times
#  14 : Ø/winter               : Ø trip lengths, Ø parking times
#  14 : Ø/winter               : Ø trip lengths, Ø parking times
#  20 : Øgermany (imported)    : from UBA/HBEFA country data template
#  21 : Ø/spring (imported)    : from UBA/HBEFA country data template
#  22 : Ø/summer (imported)    : from UBA/HBEFA country data template
#  23 : Ø/autumn (imported)    : from UBA/HBEFA country data template
#  24 : Ø/winter (imported)    : from UBA/HBEFA country data template
#  25 : Øgermany (imported v4) : from UBA/HBEFA country data template
#  30 : Øgermany (imported v4) : from UBA/HBEFA country data template
#
# --- Selected trip length only (Ø temp, Ø parking) ---
# 211: TØ, tØ, 0-1km
# 212: TØ, tØ, 1-2km
# 213: TØ, tØ, 2-3km
# 214: TØ, tØ, 3-4km
# 215: TØ, tØ, >20km
#
# --- Selected parking time only (Ø temp, Ø trip length) ---
# 251: TØ, 0-1h,  dØ
# 252: TØ, 1-2h,  dØ
# 253: TØ, 2-3h,  dØ
# 254: TØ, 3-4h,  dØ
# 255: TØ, 4-5h,  dØ
# 256: TØ, 5-6h,  dØ
# 257: TØ, 6-7h,  dØ
# 258: TØ, 7-8h,  dØ
# 259: TØ, 8-9h,  dØ
# 260: TØ, 9-10h, dØ
# 261: TØ, 10-11h,dØ
# 262: TØ, 11-12h,dØ
# 263: TØ, >12h,  dØ
#
# --- Selected parking time AND trip length (Ø temp) ---
# Format: 3XY where X = parking time bin, Y = trip length bin
# 311-343: TØ, parking 0-1h .. >12h, trip 0-1km
# 331-343: TØ, parking 0-1h .. >12h, trip 1-2km
# (pattern continues for all combinations)
#
# --- Fixed temperature + averaged other dims (1Temp) ---
# IDs ~9901-10251, step 50 per temperature:
#  9901: T-10°C, tØ, dØ
#  9951: T-5°C,  tØ, dØ
# 10001: T+0°C,  tØ, dØ
# 10051: T+5°C,  tØ, dØ
# 10101: T+10°C, tØ, dØ
# 10151: T+15°C, tØ, dØ
# 10201: T+20°C, tØ, dØ
# 10251: T+25°C, tØ, dØ
#
# --- Full pattern: fixed temp + parking time + trip length ---
# IDs 11XXX: combination of temperature, parking time bin, and trip length bin
# Format: 11 {temp_idx} {parking_idx} {trip_idx}
#
# Temperature index:  3=+20°C, 5=+10°C, 7=0°C, 9=-10°C
# Parking time index: 1=0-1h, 2=1-2h, 3=2-3h, 4=3-4h, 5=4-5h, 9=>12h
# Trip length index:  1=0-1km, 2=1-2km, 3=2-3km, 4=3-4km, 5=4-5km, 9=>5km (or N/A)
#
# Examples:
# 11311: temp_idx=3 (+20°C), parking_idx=1 (0-1h),  trip_idx=1 (0-1km)
# 11711: temp_idx=7 (  0°C), parking_idx=1 (0-1h),  trip_idx=1 (0-1km)
# 11911: temp_idx=9 (-10°C), parking_idx=1 (0-1h),  trip_idx=1 (0-1km)
# 11319: temp_idx=3 (+20°C), parking_idx=1 (0-1h),  trip_idx=9 (N/A or >5km)
# 11999: temp_idx=9 (-10°C), parking_idx=9 (>12h),  trip_idx=9 (>5km)
        


In [ ]:
# Parameters:
# emcat: emission category (hot, start, evap-soaked, evap-diurnal, evap-lr)
# yearref: reference year for emission factors e.g. 2024
# agglevel_ts: aggregation level for traffic situation (aggregate_ts, single_ts, static_ts)
def request_hbefa(emcat="hot", yearref="2024", agglevel_ts="aggregate_ts", gradients="", pollutants="", vehicles="", idts_grad="", indv_area="", indv_road=""):
    """
    Request HBEFA emission factors for specific parameters and save results as parquet file. Requires valid HBEFA credentials stored in .env file.
    Args:
        emcat: The emission category: "hot", "start", "evap-soaked", "evap-diurnal", "evap-lr".
        yearref: The reference year, e.g. "2024".
        agglevel_ts: The aggregation level: "aggregate_ts", "single_ts", "static_ts".

    Returns:
        None. Saves the requested emission factors as a parquet file in the specified path.
    """
    #Check if file already exists
    filename = f'{data_paths.EF_PATH}{yearref}_{emcat}_{agglevel_ts}.parquet'

    # Load credentials from .env file
    mail = dotenv_values(data_paths.ENV_PATH).get("HBEFA_EMAIL")
    password = dotenv_values(data_paths.ENV_PATH).get("HBEFA_PASSWORD")
    BASE_URL = "https://hbefa-server-repo.azurewebsites.net"
    session = requests.Session()
    get_response = session.get(f"{BASE_URL}/login")

    # Get CSRF token directly from the form HTML (more reliable)
    csrf_token = re.search(r'id="csrf_token" name="csrf_token".*?value="(.*?)"', get_response.text).group(1)
    print("CSRF token:", csrf_token)

    login_response = session.post(
        f"{BASE_URL}/login",
        data={
            "email": mail,
            "password": password,
            "csrf_token": csrf_token,
            "submit": "Login"
        },
        headers={"X-CSRFToken": csrf_token, "Referer": f"{BASE_URL}/login"}
    )

    print("Login status:", login_response.status_code)
    print("Session cookie:", session.cookies.get("session")[:40], "...")

    if(agglevel_ts == "aggregate_ts"):
         payload = {
        "country": "D",
        "pollutant": pollutants,
        "emcat": emcat,
        "hbversion_int": "501006",
        "agglevel_ts": agglevel_ts,
        "idvehcat": vehicles, 
        "wgt": "True", # Whether to weight emission factors by fleet composition (True/False).
        "idtraffic_scen": "48",
        "yearref": yearref,
        "agglevel_fleet": "vehcat",
        "agglevel_energy": "none",
        "nocorr": "False",
        "lang": "en",
        "col_selection": "standard",
        "col_titles": "speaking",
        "load_in_rows": "False",
        "agg_cols": "False",
        "verbose": "False",
        "test_outputs": "False",
        "calc_wtt": "True", #Doesn't exist for HBEFA 5.1, but doesn't cause an error if included in the request
        "idenergymix_scen": "3",
        "idpatternambientcond": "9901,9951,10001,10051,10101,10151,10201,10251", # Ambient conditions: 8 temperature levels from -10°C to +25°C (in 5°C steps),each with average parking time and average trip length (tØ/dØ)
    }
    else:
        # Now submit the job using the session
        payload = {
            "country": "D",
            "pollutant": pollutants,
            "emcat": emcat,
            "hbversion_int": "501006",
            "agglevel_ts": agglevel_ts,
            "idvehcat": vehicles, 
            "wgt": "True", # Whether to weight emission factors by fleet composition (True/False).
            "idtraffic_scen": "48",
            "idtsgrad": idts_grad,
            "yearref": yearref,
            "agglevel_fleet": "vehcat",
            "agglevel_energy": "none",
            "nocorr": "False",
            "lang": "en",
            "col_selection": "standard",
            "col_titles": "speaking",
            "load_in_rows": "False",
            "agg_cols": "False",
            "verbose": "False",
            "test_outputs": "False",
            "calc_wtt": "True", #Doesn't exist for HBEFA 5.1, but doesn't cause an error if included in the request
            "idenergymix_scen": "3",
            "idgrad": gradients,
            "idarea": indv_area,
            "idroadtype": indv_road,
            "idpatternambientcond": "9901,9951,10001,10051,10101,10151,10201,10251", # Ambient conditions: 8 temperature levels from -10°C to +25°C (in 5°C steps),each with average parking time and average trip length (tØ/dØ)
        }
    print(payload)
    response = session.post(f"{BASE_URL}/efa-async", json=payload)
    print("Job submitted:", response.status_code)
    print("Job response:", response.text)

    # Poll for result
    task_id = response.json().get("task_id")
    for i in range(500):
        result = session.get(f"{BASE_URL}/efa-async/{task_id}")
        print(f"Poll {i + 1}: {result.status_code} - {result.text[:100]}")

        data = result.json()
        status = data.get("status", "")

        # Keep polling while task is queued or running
        if result.status_code == 200:
            print("Raw result:")
            print(data)
            emission_factors = data.get("Emission factors")
            print(emission_factors)
            df = pd.read_json(StringIO(emission_factors))
            #df.to_json('test.json')
            #df.to_csv('test.csv', index=True, index_label="index")
            df.to_parquet(f'{filename}', index=True)
            break
        if result.status_code != 202:
            print("Error retrieving result:", result.status_code, result.text)
            break
        time.sleep(3)
    else:
        print("Timed out waiting for result")
# Press the green button in the gutter to run the script.


In [46]:
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        request_hbefa(emcat_dropdown.value, yearref_dropdown.value, agglevel_dropdown.value,ui_gradients.result, 
                  ui_pollutants.result, ui_vehicles.result, ui_vehicles.result, ui_indv_area.result, ui_indv_roadtype.result)

print(select_pollutants())
ui_gradients = interactive(
    select_gradient,
    grad_0   = widgets.Checkbox(value=True, description="0%"),
    grad_pm2 = widgets.Checkbox(value=True, description="+/-2%"),
    grad_pm4 = widgets.Checkbox(value=True, description="+/-4%"),
    grad_pm6 = widgets.Checkbox(value=True, description="+/-6%"),
    grad_m6  = widgets.Checkbox(value=True, description="-6%"),
    grad_m4  = widgets.Checkbox(value=True, description="-4%"),
    grad_m2  = widgets.Checkbox(value=True, description="-2%"),
    grad_p2  = widgets.Checkbox(value=True, description="+2%"),
    grad_p4  = widgets.Checkbox(value=True, description="+4%"),
    grad_p6  = widgets.Checkbox(value=True, description="+6%"),
)


ui_pollutants = interactive(
    select_pollutants,
    hc              = widgets.Checkbox(value=False, description="HC"),
    co              = widgets.Checkbox(value=True,  description="CO"),
    nox             = widgets.Checkbox(value=True,  description="NOx"),
    no2             = widgets.Checkbox(value=True,  description="NO2"),
    co2_rep         = widgets.Checkbox(value=True,  description="CO2(rep)"),
    co2_total       = widgets.Checkbox(value=True,  description="CO2(total)"),
    pm10_ex         = widgets.Checkbox(value=True,  description="PM10-ex"),
    pn23_ex         = widgets.Checkbox(value=False, description="PN23-ex"),
    ch4             = widgets.Checkbox(value=True,  description="CH4"),
    nmhc            = widgets.Checkbox(value=False, description="NMHC"),
    pb              = widgets.Checkbox(value=False, description="Pb"),
    so2             = widgets.Checkbox(value=False, description="SO2"),
    n2o             = widgets.Checkbox(value=False, description="N2O"),
    nh3             = widgets.Checkbox(value=False, description="NH3"),
    zn_ex           = widgets.Checkbox(value=False, description="Zn-ex"),
    zn_nx           = widgets.Checkbox(value=False, description="Zn-nx"),
    cd_ex           = widgets.Checkbox(value=False, description="Cd-ex"),
    cd_nx           = widgets.Checkbox(value=False, description="Cd-nx"),
    pm10_nx         = widgets.Checkbox(value=True,  description="PM10-nx"),
    benzene         = widgets.Checkbox(value=False, description="Benzene"),
    toluene         = widgets.Checkbox(value=False, description="Toluene"),
    xylene          = widgets.Checkbox(value=False, description="Xylene"),
    fc              = widgets.Checkbox(value=False, description="FC"),
    ec              = widgets.Checkbox(value=False, description="EC"),
    pm25_ex         = widgets.Checkbox(value=True,  description="PM2.5-ex"),
    bc_ex           = widgets.Checkbox(value=True,  description="BC-ex"),
    pm25_nx         = widgets.Checkbox(value=True,  description="PM2.5-nx"),
    bc_nx           = widgets.Checkbox(value=True,  description="BC-nx"),
    co2e            = widgets.Checkbox(value=False, description="CO2e"),
    we_pos          = widgets.Checkbox(value=False, description="WE-pos"),
    hcho            = widgets.Checkbox(value=False, description="HCHO"),
    ch3cho          = widgets.Checkbox(value=False, description="CH3CHO"),
    hnco            = widgets.Checkbox(value=False, description="HNCO"),
    hno2            = widgets.Checkbox(value=False, description="HNO2"),
    pm10_nx_tyre    = widgets.Checkbox(value=False, description="PM10-nx-tyre"),
    pm10_nx_brake   = widgets.Checkbox(value=False, description="PM10-nx-brake"),
    pm10_nx_road    = widgets.Checkbox(value=False, description="PM10-nx-road"),
    pm10_nx_resusp  = widgets.Checkbox(value=False, description="PM10-nx-resusp"),
    pm25_nx_tyre    = widgets.Checkbox(value=False, description="PM2.5-nx-tyre"),
    pm25_nx_brake   = widgets.Checkbox(value=False, description="PM2.5-nx-brake"),
    pm25_nx_road    = widgets.Checkbox(value=False, description="PM2.5-nx-road"),
    pm25_nx_resusp  = widgets.Checkbox(value=False, description="PM2.5-nx-resusp"),
    pn23_nx_brake   = widgets.Checkbox(value=False, description="PN23-nx-brake"),
    pn23_nx_road    = widgets.Checkbox(value=False, description="PN23-nx-road"),
    pn23_nx_resusp  = widgets.Checkbox(value=False, description="PN23-nx-resusp"),
    pn23_nx         = widgets.Checkbox(value=False, description="PN23-nx"),
)
ui_vehicles = interactive(select_vehicles, 
                          pc = widgets.Checkbox(value=True, description="PC"),
                          lcv = widgets.Checkbox(value=True, description="LCV"),
                          hgv = widgets.Checkbox(value=True, description="HGV"),
                          coach = widgets.Checkbox(value=True, description="Coach"),
                          bus = widgets.Checkbox(value=False, description="Urban Bus"),
                          motorcycle = widgets.Checkbox(value=True, description="Motor Cycle")
)
ui_idtsgrad = interactive(
    select_tsgrad,
    d_mw_2024         = widgets.Checkbox(value=False,  description="D Ø-MW UBA 2024"),
    d_rural_2024      = widgets.Checkbox(value=False,  description="D Ø-Rural UBA 2024"),
    d_urban_2024      = widgets.Checkbox(value=False,  description="D Ø-Urban UBA 2024"),
    d_all_2024        = widgets.Checkbox(value=True,  description="D Ø UBA 2024"),

    d_mw_2023         = widgets.Checkbox(value=False, description="D Ø-MW UBA 2023"),
    d_rural_2023      = widgets.Checkbox(value=False, description="D Ø-Rural UBA 2023"),
    d_urban_2023      = widgets.Checkbox(value=False, description="D Ø-Urban UBA 2023"),
    d_all_2023        = widgets.Checkbox(value=False, description="D Ø UBA 2023"),

    d_all_2022        = widgets.Checkbox(value=False, description="D Ø UBA 2022"),
    d_mw_2022         = widgets.Checkbox(value=False, description="D Ø-MW UBA 2022"),
    d_rural_2022      = widgets.Checkbox(value=False, description="D Ø-Rural UBA 2022"),
    d_urban_2022      = widgets.Checkbox(value=False, description="D Ø-Urban UBA 2022"),

    d_mw_2021         = widgets.Checkbox(value=False, description="D Ø-MW UBA 2021"),
    d_rural_2021      = widgets.Checkbox(value=False, description="D Ø-Rural UBA 2021"),
    d_urban_2021      = widgets.Checkbox(value=False, description="D Ø-Urban UBA 2021"),
    d_all_2021        = widgets.Checkbox(value=False, description="D Ø UBA 2021"),
    d_all_2021_det    = widgets.Checkbox(value=False, description="D Ø UBA 2021 detailed"),
    d_mw_2021_korr    = widgets.Checkbox(value=False, description="D Ø-MW UBA 2021 korr"),
    d_rural_2021_korr = widgets.Checkbox(value=False, description="D Ø-Rural UBA 2021 korr"),
    d_urban_2021_korr = widgets.Checkbox(value=False, description="D Ø-Urban UBA 2021 korr"),
    d_all_2021_korr   = widgets.Checkbox(value=False, description="D Ø UBA 2021 korr"),

    de_mw_ifeu        = widgets.Checkbox(value=False, description="Germany Motorway (IFEU)"),
    de_rural_ifeu     = widgets.Checkbox(value=False, description="Germany Rural (IFEU)"),
    de_urban_ifeu     = widgets.Checkbox(value=False, description="Germany Urban (IFEU)"),
    de_all_ifeu       = widgets.Checkbox(value=False, description="Germany all (IFEU)"),
)

ui_indv_area = interactive(select_indv_scen_area,
                           urban = widgets.Checkbox(value=True, description="Urban"),
                           rural = widgets.Checkbox(value=False, description="Rural"))

ui_indv_roadtype = interactive(
    select_indv_scen_roadtype,
    motorway_nat       = widgets.Checkbox(value=True,  description="Motorway-Nat."),
    motorway_city      = widgets.Checkbox(value=False, description="Motorway-City"),
    semi_motorway      = widgets.Checkbox(value=False, description="Semi-Motorway"),
    primary_nat        = widgets.Checkbox(value=True,  description="Primary-Nat."),
    primary_city       = widgets.Checkbox(value=True,  description="Primary-City"),
    distributor        = widgets.Checkbox(value=True,  description="Distributor"),
    distributor_sin    = widgets.Checkbox(value=False, description="Distributor (sinuous)"),
    local              = widgets.Checkbox(value=True,  description="Local/Collector"),
    local_sin          = widgets.Checkbox(value=False, description="Local/Collector (sinuous)"),
    access_residential = widgets.Checkbox(value=True,  description="Access-Residential"),
)
emcat_dropdown = widgets.Dropdown(options={"Hot Emissions": "hot", "Start Emissions": "start", "Evap-Soaked": "evap-soaked", "Evap-Diurnal": "evap-diurnal", "Evap-LR": "evap-lr"})
yearref_dropdown = widgets.Dropdown(options=["2024", "2025"])
agglevel_dropdown = widgets.Dropdown(options=["aggregate_ts", "single_ts", "static_ts"])

accordion = Accordion(children=[
    ui_gradients,    # panel 0
    ui_pollutants,   # panel 1
    ui_vehicles,
    ui_idtsgrad,
    ui_indv_area,
    ui_indv_roadtype,
    emcat_dropdown,
    yearref_dropdown,
    agglevel_dropdown
])
accordion.set_title(0, "Gradients")
accordion.set_title(1, "Pollutants")
accordion.set_title(2, "Vehicle Types")
accordion.set_title(3, "Aggregated Traffic Situation Pattern")
accordion.set_title(4, "Individual Traffic Situation Area")
accordion.set_title(5, "Individual Traffic Situation Road Type")
accordion.set_title(6, "Emission Category")
accordion.set_title(7, "Reference Year")
accordion.set_title(8, "Aggregation Level")
accordion.selected_index = None
print(select_indv_scen_roadtype())

button = widgets.Button(description="Run")
button._click_handlers.callbacks.clear()
button.on_click(on_click)
display(accordion, button, output)


HC,CO,NOx,NO2,CO2(rep),CO2(total),PM10-ex,PN23-ex,CH4,NMHC,Pb,SO2,N2O,NH3,Zn-ex,Zn-nx,Cd-ex,Cd-nx,PM10-nx,Benzene,Toluene,Xylene,FC,EC,PM2.5-ex,BC-ex,PM2.5-nx,BC-nx,CO2e,WE-pos,HCHO,CH3CHO
10,20,21,30,40,50


Accordion(children=(interactive(children=(Checkbox(value=True, description='0%'), Checkbox(value=True, descrip…

Button(description='Run', style=ButtonStyle())

Output()